In [1]:
import napari
import os
import numpy as np
from skimage import io
from scipy.spatial import distance, KDTree
from napari import layers
import pandas as pd

#from ultralytics import YOLO

In [2]:
viewer = napari.Viewer()

In [4]:
root = r'C:\Users\johamuel\Desktop\local_data\tiles'

viewer.layers.clear()
list_of_files = [os.path.join(root, file) for file in os.listdir(root) if not 'converted' in file]
_ = viewer.open(list_of_files, plugin='napari-autoscape')

In [6]:
viewer.layers[0].plane

SlicingPlane(normal=(np.float64(1.0), np.float64(0.0), np.float64(0.0)), position=(0.0, 0.0, 0.0), thickness=1.0)

In [7]:
model = YOLO(r'D:\Git\autoSCAPE\docs\notebooks\runs\detect\train22\weights\best.pt')

In [8]:
rectangles = []
features = pd.DataFrame()
for layer in viewer.layers:
    if not isinstance(layer, layers.Image):
        continue

    io.imsave('image.png', layer.data[0][0].compute())
    prediction = model('image.png')
    for obj_class, box, conf in zip(prediction[0].boxes.cls, prediction[0].boxes.xywhn, prediction[0].boxes.conf):
        cls = int(obj_class)
        box = box.detach().cpu().numpy()

        z0 = layer.translate[0]
        y0 = layer.translate[1]
        x0 = layer.translate[2]
        img_size = layer.data.shape * layer.scale
        y0_box = box[1] * img_size[1]
        x0_box = box[0] * img_size[2]
        w = box[2] * img_size[2]
        h = box[3] * img_size[1]

        rectangle_corners = np.array([
            [z0, y0 + y0_box - h/2, x0 + x0_box - w/2],
            [z0, y0 + y0_box - h/2, x0 + x0_box + w - w/2],
            [z0, y0 + y0_box - h/2 + h, x0 + x0_box + w - w/2],
            [z0, y0 + y0_box - h/2 + h, x0 + x0_box - w/2]
        ])
        rectangle_corners[:, 1:] += box[:2] * layer.scale[1:]

        _features = pd.DataFrame({
            'y0': [y0_box],
            'x0': [x0_box],
            'w': [w],
            'h': [h],
            'cls': [cls],
            'conf': [conf.detach().cpu().numpy()],
            'area': [w * h],
        })
        features = pd.concat([features, _features], ignore_index=True)

    rectangles.append(rectangle_corners)


image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 512x512 1 cell, 35.0ms
Speed: 2.1ms preprocess, 35.0ms inference, 0.5ms postprocess per image at shape (1, 3, 512, 512)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 512x512 5 cells, 20.7ms
Speed: 1.4ms preprocess, 20.7ms inference, 0.5ms postprocess per image at shape (1, 3, 512, 512)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 512x512 7 cells, 21.2ms
Speed: 1.3ms preprocess, 21.2ms inference, 0.5ms postprocess per image at shape (1, 3, 512, 512)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 512x512 3 cells, 1 spheroid, 21.0ms
Speed: 1.4ms preprocess, 21.0ms inference, 0.6ms postprocess per image at shape (1, 3, 512, 512)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 512x512 3 cells, 20.6ms
Speed: 1.2ms preprocess, 20.6ms inference, 0.5ms postprocess per image at shape (1, 3, 512, 512)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 512x512 1 spheroid, 21.6ms
Speed: 1.3ms preprocess, 21.6ms i

In [9]:
viewer.add_shapes(rectangles, features=features, edge_width=10, face_color='transparent',)

<Shapes layer 'rectangles' at 0x2393653a850>

In [ ]:
rectangles

[array([[     1925.1,       11072,       30796],
        [     1925.1,       11072,       30813],
        [     1925.1,       11085,       30813],
        [     1925.1,       11085,       30796]]),
 array([[     1925.1,       10657,       30703],
        [     1925.1,       10657,       30723],
        [     1925.1,       10675,       30723],
        [     1925.1,       10675,       30703]]),
 array([[     1925.1,       10378,       30715],
        [     1925.1,       10378,       30730],
        [     1925.1,       10391,       30730],
        [     1925.1,       10391,       30715]]),
 array([[     1925.1,      9838.4,       30586],
        [     1925.1,      9838.4,       30686],
        [     1925.1,      9958.7,       30686],
        [     1925.1,      9958.7,       30586]]),
 array([[     1925.1,      9579.5,       30635],
        [     1925.1,      9579.5,       30651],
        [     1925.1,      9594.9,       30651],
        [     1925.1,      9594.9,       30635]]),
 array([[ 

c:\Users\Johan\mambaforge\envs\clusters-plotter\Lib\site-packages\biaplotter\colormap.py:34: UserWarning: Categorical colormap detected. Setting categorical=True. If the colormap is continuous, set categorical=False explicitly.
  warnings.warn(
c:\Users\Johan\mambaforge\envs\clusters-plotter\Lib\site-packages\biaplotter\artists.py:326: UserWarning: Color indices must be integers for categorical colormap. Change `overlay_colormap` to a continuous colormap or set `color_indices` to integers.
  warnings.warn(
c:\Users\Johan\mambaforge\envs\clusters-plotter\Lib\site-packages\biaplotter\artists.py:326: UserWarning: Color indices must be integers for categorical colormap. Change `overlay_colormap` to a continuous colormap or set `color_indices` to integers.
  warnings.warn(
c:\Users\Johan\mambaforge\envs\clusters-plotter\Lib\site-packages\biaplotter\artists.py:326: UserWarning: Color indices must be integers for categorical colormap. Change `overlay_colormap` to a continuous colormap or set 

In [26]:
from napari_clusters_plotter import PlotterWidget
plotter = PlotterWidget(viewer)
viewer.window.add_dock_widget(plotter, name='Clusters Plotter', area='right')

ModuleNotFoundError: No module named 'matplotlib.colorbar'

In [10]:
import matplotlib.pyplot as plt

plt.colorbar

ModuleNotFoundError: No module named 'matplotlib.colorbar'

In [ ]:
well_grid = viewer.layers['Well grid']
image_fused = viewer.layers['fused :: default_channel']

for idx, shape in enumerate(well_grid.data):

    row = well_grid.features['row'][idx]
    col = well_grid.features['column'][idx]

    # shape in pixel coordinates
    shape_px = (shape - image_fused.translate) / image_fused.scale
    limits = np.stack([
        np.min(shape_px, axis=0),
        np.max(shape_px, axis=0)
    ]).astype(int)[:, 1:]

    crop = image_fused.data[0][0,
        limits[0, 0] : limits[1, 0],
        limits[0, 1] : limits[1, 1],
    ]

    ezomero.post_image(
        conn=conn,
        image=crop.to_numpy()[None, None, None, :],
        dim_order='tczyx',
        dataset_id=408,
        image_name=f'well_{int(row):02d}_{int(col):02d}',
        across_groups=False
    )
    #ezomero.post_image

    # translate_crop = image_fused.translate[1:] + limits[0, :] * image_fused.scale[1:]
    # layer = viewer.add_image(crop,
    #      translate=translate_crop,
    #      scale=image_fused.scale[1:],
    #      name='crop'
    #      )
    # layer.bounding_box.visible = True
    # break
